In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/fake-news-detection/true.csv
/kaggle/input/fake-news-detection/fake.csv


## Setting Up the Environment

In [2]:
!pip install torch torch-geometric spektral
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.4.0+${CUDA}.html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 9.7 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.4.0+.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 8.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for torch-scatter: filename=torch_scatter-2.1.2-cp310-cp310-linux_x86_64.whl size=3876478 sha256=fc7a83e81c69fc659fa5b09c56443d3541790c15c2b05d4c18bb4797fde8617e
  Stored in directory: /root/.cache/pip/wheels/92/f1/2b/3b46d54b134259f58c8363568569053248040859b1a145b3ce
  Created wheel for torch-sparse: filename=torch_sparse-0.6.18-cp310-cp310-linux_x86_64.whl size=2713599 sha256=d43ca899b997ef2a4b1712e4a66d0fe9dd205c706e9a26b354ef1715b32

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
import networkx as nx
from spektral.utils import normalized_adjacency

## Data Loading and Preprocessing

In [4]:
# Load datasets
fake_df = pd.read_csv('/kaggle/input/fake-news-detection/fake.csv')
true_df = pd.read_csv('/kaggle/input/fake-news-detection/true.csv')

# Add labels
fake_df['label'] = 1  # 1 for fake
true_df['label'] = 0  # 0 for true

In [5]:
print(fake_df.shape)
print(true_df.shape)

# Combine datasets
df = pd.concat([fake_df, true_df], ignore_index=True)

(23481, 5)
(21417, 5)


In [6]:
import string

def preprocess_text(text):
    # Convert the text to lowercase
    text = text.lower()
    # Remove all punctuation marks
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove extra whitespace (leading, trailing, and multiple spaces)
    text = ' '.join(text.split())
    return text

# Apply text preprocessing to the 'text' and 'title' columns
df['text'] = df['text'].astype(str).apply(preprocess_text)
df['title'] = df['title'].astype(str).apply(preprocess_text)

# Combine the 'title' and 'text' columns into a single 'content' column
df['content'] = df['title'] + " " + df['text']

# Sample a subset of the data to manage computational resources
sample_size = 6000
df_sample = df.sample(n=sample_size, random_state=42).reset_index(drop=True)

# Print the number of sampled articles
print(f"Sampled {sample_size} articles for analysis.")

Sampled 6000 articles for analysis.


In [7]:
# Extract the 'label' column as the target variable (y_data)
y_data = df_sample['label'].values  

# Drop the 'label' column from the DataFrame to avoid leakage during model training
df_sample.drop(['label'], axis=1, inplace=True)

# Display the extracted target variable
y_data

array([1, 0, 0, ..., 1, 1, 0])

In [8]:
df_sample.head(10)

,title,text,subject,date,content
0,ben stein calls out 9th circuit court committe...,21st century wire says ben stein reputable pro...,US_News,"February 13, 2017",ben stein calls out 9th circuit court committe...
1,trump drops steve bannon from national securit...,washington reuters us president donald trump r...,politicsNews,"April 5, 2017",trump drops steve bannon from national securit...
2,puerto rico expects us to lift jones act shipp...,reuters puerto rico governor ricardo rossello ...,politicsNews,"September 27, 2017",puerto rico expects us to lift jones act shipp...
3,oops trump just accidentally confirmed he leak...,on monday donald trump once again embarrassed ...,News,"May 22, 2017",oops trump just accidentally confirmed he leak...
4,donald trump heads for scotland to reopen a go...,glasgow scotland reuters most us presidential ...,politicsNews,"June 24, 2016",donald trump heads for scotland to reopen a go...
5,paul ryan responds to dem’s sitin on gun contr...,on wednesday democrats took a powerful stance ...,News,"June 22, 2016",paul ryan responds to dem’s sitin on gun contr...
6,awesome diamond and silk rip into the press “w...,president trump s rally in fl on saturday was ...,Government News,"Feb 19, 2017",awesome diamond and silk rip into the press “w...
7,stand up and cheer ukip party leader slams ger...,he s been europe s version of the outspoken te...,left-news,"Mar 8, 2016",stand up and cheer ukip party leader slams ger...
8,north korea shows no sign it is serious about ...,washington reuters the state department said w...,worldnews,"December 13, 2017",north korea shows no sign it is serious about ...
9,trump signals willingness to raise us minimum ...,this version of the story corrects the figure ...,politicsNews,"May 4, 2016",trump signals willingness to raise us minimum ...


## Feature Extraction

In [9]:
# Initialize the TF-IDF vectorizer with a limit of 2000 features
vectorizer = TfidfVectorizer(max_features=2000)  
# Transform the 'content' column into a TF-IDF feature matrix
tfidf_features = vectorizer.fit_transform(df_sample['content']).toarray()

from sklearn.preprocessing import StandardScaler

# Normalize the TF-IDF features using StandardScaler
scaler = StandardScaler()
tfidf_features = scaler.fit_transform(tfidf_features)

# Print the shape of the resulting TF-IDF feature matrix
print(f"TF-IDF feature matrix shape: {tfidf_features.shape}")

TF-IDF feature matrix shape: (6000, 2000)


## Constructing the Graph

In [10]:
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

k = 5  # Number of nearest neighbors

# Initialize NearestNeighbors with cosine similarity as the distance metric
nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='auto', metric='cosine').fit(tfidf_features)

# Find the k+1 nearest neighbors (including the point itself)
distances, indices = nbrs.kneighbors(tfidf_features)

# Initialize lists to store the rows and columns of the adjacency matrix
rows = []
cols = []

# Build the adjacency matrix by adding edges between the nodes and their neighbors
for i in range(tfidf_features.shape[0]):
    for j in range(1, k+1):  # Skip the first neighbor (itself)
        neighbor = indices[i][j]
        rows.append(i)
        cols.append(neighbor)

# Create a sparse adjacency matrix where each edge has a value of 1
data = np.ones(len(rows))
adjacency_sparse = csr_matrix((data, (rows, cols)), shape=(tfidf_features.shape[0], tfidf_features.shape[0]))

# Symmetrize the adjacency matrix to ensure the graph is undirected (bidirectional edges)
adjacency_sparse = adjacency_sparse.maximum(adjacency_sparse.transpose())

# Print the number of nodes and edges in the sparse adjacency matrix
print(f"Number of nodes: {adjacency_sparse.shape[0]}")
print(f"Number of edges: {adjacency_sparse.count_nonzero()}")

Number of nodes: 6000
Number of edges: 45085


In [11]:
# Normalize the adjacency matrix to improve convergence during training
A_norm_sparse = normalized_adjacency(adjacency_sparse, symmetric=True)

# Indicate that the adjacency matrix has been normalized
print("Adjacency matrix normalized.")

Adjacency matrix normalized.


## Prepare Tensors for Torch Modeling

In [12]:
import tensorflow as tf
import numpy as np
from spektral.utils import normalized_adjacency
from scipy.sparse import csr_matrix

# Use your actual normalized adjacency matrix (A_norm_sparse)
A_norm_sparse = csr_matrix(A_norm_sparse)  # Convert to CSR format if not already

# Convert the sparse matrix (csr_matrix) to a SparseTensor in TensorFlow
A_sparse_tensor = tf.sparse.from_dense(A_norm_sparse.toarray())  # Convert to dense, then to SparseTensor

# Normalize the adjacency matrix using Spektral (works with sparse matrices)
A_normalized = normalized_adjacency(A_norm_sparse)

# Convert the normalized sparse matrix to a dense numpy array
A_dense = A_normalized.toarray()  # Convert the csr_matrix to a dense array

# Convert the dense numpy array to a TensorFlow tensor
A_tensor = tf.convert_to_tensor(A_dense, dtype=tf.float32)

# Ensure X_tensor is a TensorFlow tensor and cast it to float32
X_tensor = tf.convert_to_tensor(tfidf_features, dtype=tf.float32)  # Use actual node features

# Convert labels to tensor (ensure the data type is float32 for consistency)
y_tensor = tf.convert_to_tensor(y_data, dtype=tf.float32)

# Optionally, check the shapes of the tensors for verification
print(A_tensor.shape)
print(X_tensor.shape)
print(y_tensor.shape)

(6000, 6000)
(6000, 2000)
(6000,)


## Splitting Data into Train and Test Sets

In [13]:
idx = np.arange(df_sample.shape[0])  # Create an array of indices for the dataset
# Split the indices into training and testing sets (80% train, 20% test)
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=42)

# Create boolean masks for the training set (True for train, False for others)
train_mask = np.zeros(df_sample.shape[0], dtype=bool)
train_mask[idx_train] = True

# Create boolean masks for the testing set (True for test, False for others)
test_mask = np.zeros(df_sample.shape[0], dtype=bool)
test_mask[idx_test] = True

# Print the number of training and testing samples
print(f"Training samples: {train_mask.sum()}")
print(f"Testing samples: {test_mask.sum()}")

Training samples: 4800
Testing samples: 1200


## Train the Model

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
import torch_sparse

# Convert data to PyTorch tensors
X_tensor = torch.tensor(X_tensor.numpy(), dtype=torch.float32)  # Node features (X)
A_tensor = torch.tensor(A_tensor.numpy(), dtype=torch.float32)  # Adjacency matrix (A)
y_tensor = torch.tensor(y_tensor.numpy(), dtype=torch.long)    # Labels (y)

# PyTorch Geometric requires sparse adjacency matrix format, so convert it
from torch_sparse import coalesce
from torch_geometric.utils import dense_to_sparse

# Convert adjacency matrix to sparse COO format
edge_index, _ = dense_to_sparse(A_tensor)

# Create a PyTorch Geometric data object that contains the node features, edge indices, and labels
data = Data(x=X_tensor, edge_index=edge_index, y=y_tensor)

# Define the Graph Convolutional Network (GCN) model
class GCNModel(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(GCNModel, self).__init__()
        self.conv1 = GCNConv(in_channels, 128)  # First GCN layer with 128 output channels
        self.conv2 = GCNConv(128, out_channels)  # Second GCN layer with output channels (final prediction)
    
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)  # Apply first GCN layer
        x = torch.relu(x)  # Apply ReLU activation function
        x = self.conv2(x, edge_index)  # Apply second GCN layer
        return x

# Instantiate the GCN model, set the input channels and output classes (2 classes: real or fake)
model = GCNModel(in_channels=X_tensor.shape[1], out_channels=2)

# Define the optimizer and the loss function (CrossEntropyLoss for classification)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

# Training loop
epochs = 100
for epoch in range(epochs):
    model.train()  # Set the model to training mode
    optimizer.zero_grad()  # Zero the gradients for each epoch
    
    # Perform a forward pass
    out = model(data)
    
    # Calculate the loss using the true labels (data.y) and predicted outputs (out)
    loss = loss_fn(out, data.y)
    
    # Perform backward pass and optimization
    loss.backward()
    optimizer.step()
    
    # Print the loss every 10 epochs
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

Epoch 0, Loss: 0.7890
Epoch 10, Loss: 0.2792
Epoch 20, Loss: 0.2165
Epoch 30, Loss: 0.1889
Epoch 40, Loss: 0.1686
Epoch 50, Loss: 0.1525
Epoch 60, Loss: 0.1383
Epoch 70, Loss: 0.1250
Epoch 80, Loss: 0.1126
Epoch 90, Loss: 0.1008


## Evaluate on Test Set

In [15]:
from sklearn.metrics import classification_report

# Make predictions
model.eval()
with torch.no_grad():
    out = model(data)
    _, predicted = torch.max(out, dim=1)  # Get the class with the highest score

# Generate the classification report
print(classification_report(data.y.cpu(), predicted.cpu(), target_names=["Class 0", "Class 1"]))

              precision    recall  f1-score   support

     Class 0       0.97      0.97      0.97      2865
     Class 1       0.97      0.97      0.97      3135

    accuracy                           0.97      6000
   macro avg       0.97      0.97      0.97      6000
weighted avg       0.97      0.97      0.97      6000



## Save our Model

In [16]:
# Save the trained model
torch.save(model.state_dict(), 'gcn_model.pth')